## Configurações Globais (config.py)

In [ ]:
%%writefile config.py
import pygame

pygame.init()

# =============================
# CONFIGURAÇÃO
# =============================

pygame.init()

TAMANHO = 24
LINHAS = 30
COLUNAS = 30

LARGURA = COLUNAS * TAMANHO + 300
ALTURA = LINHAS * TAMANHO

tela = pygame.display.set_mode((LARGURA, ALTURA))
pygame.display.set_caption("Planejamento de Rotas - Drone Agrícola")

clock = pygame.time.Clock()
fonte = pygame.font.SysFont("Arial", 16)

E_MAX = 100  # energia máxima do drone
CUSTO_MOVIMENTO = 1  # custo de energia para cada movimento
bateria = E_MAX
largura_barra = 120


DIRECOES = [(1, 0), (-1, 0), (0, 1), (0, -1)]

# =============================
# CORES
# =============================

BRANCO = (255, 255, 255)
PRETO = (30, 30, 30)
AZUL = (80, 140, 255)
AMARELO = (255, 220, 0)
VERDE = (0, 200, 100)
VERMELHO = (255, 50, 50)
CINZA = (200, 200, 200)

## Geração e Renderização do Mapa (mapa.py)

In [ ]:
%%writefile mapa.py
import random

import pygame
from config import LINHAS, COLUNAS, TAMANHO, BRANCO, PRETO, AZUL, AMARELO, VERDE, VERMELHO, CINZA


# =============================
# POSIÇÕES
# =============================


start = (1, 1)  # canto da área azul
goal = (10, 18)
# =============================
# GERAR MAPA
# =============================


def gerar_mapa():

    grid = [[0 for _ in range(COLUNAS)] for _ in range(LINHAS)]

    # região retangular
    for i in range(10):
        for j in range(12):
            grid[i][j] = 2

    # região circular
    cx, cy = 5, 18
    r = 5

    for i in range(LINHAS):
        for j in range(COLUNAS):
            if (i - cx) ** 2 + (j - cy) ** 2 <= r**2:
                grid[i][j] = 3

    # obstáculos
    for i in range(LINHAS):
        for j in range(COLUNAS):

            if (grid[i][j] == 2 or grid[i][j] == 3) and random.random() < 0.15:
                grid[i][j] = 1
    #    detalhes do mapa
    for i in range(LINHAS):
        for j in range(COLUNAS):

            if grid[i][j] == 0 and random.random() < 0.18:
                grid[i][j] = 6

    for i in range(LINHAS):
        for j in range(COLUNAS):

            if grid[i][j] == 0 and random.random() < 0.18:
                grid[i][j] = 7
    # garantir start e goal livres
    grid[start[0]][start[1]] = 0
    grid[goal[0]][goal[1]] = 0

    return grid


def desenhar(tela,grid, drone=None):

    tela.fill(BRANCO)

    for i in range(LINHAS):
        for j in range(COLUNAS):

            rect = pygame.Rect(j * TAMANHO, i * TAMANHO, TAMANHO, TAMANHO)
            match grid[i][j]:
                case 0:
                    pygame.draw.rect(tela, [93,101,50], rect)
                case 1:
                    pygame.draw.rect(tela, [139,69,19], rect)
                case 2:
                    pygame.draw.rect(tela, AZUL, rect)
                case 3:
                    pygame.draw.rect(tela, AMARELO, rect)
                case 6:
                    pygame.draw.rect(tela, [0,100,0], rect)
                case 7:
                    pygame.draw.rect(tela, [120,250,120], rect)

    

    # start
    pygame.draw.circle(
        tela,
        (0, 255, 0),
        (start[1] * TAMANHO + TAMANHO // 2, start[0] * TAMANHO + TAMANHO // 2),
        TAMANHO // 3,
    )

    # goal
    pygame.draw.circle(
        tela,
        (255, 0, 0),
        (goal[1] * TAMANHO + TAMANHO // 2, goal[0] * TAMANHO + TAMANHO // 2),
        TAMANHO // 3,
    )

    if drone:

        x, y = drone

        pygame.draw.circle(
            tela,
            VERMELHO,
            (y * TAMANHO + TAMANHO // 2, x * TAMANHO + TAMANHO // 2),
            TAMANHO // 3,
        )

## Inteligência Artificial e Buscas (algoritmos.py)

In [ ]:
%%writefile algoritmos.py
import heapq
from config import LINHAS, COLUNAS, DIRECOES, CUSTO_MOVIMENTO, E_MAX, bateria


# =============================
# HEURÍSTICA
# =============================


def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


# =============================
# VIZINHOS
# =============================


def neighbors(grid, n):

    x, y = n
    result = []

    for dx, dy in DIRECOES:

        nx = x + dx
        ny = y + dy

        if 0 <= nx < LINHAS and 0 <= ny < COLUNAS:

            if grid[nx][ny] != 1:
                result.append((nx, ny))

    return result


# =============================
# CUSTO
# =============================


def custo(grid, x, y):

    if grid[x][y] == 2:
        return CUSTO_MOVIMENTO * 0.5
    elif grid[x][y] == 3:
        return CUSTO_MOVIMENTO * 1.5
    else:
        return CUSTO_MOVIMENTO

# A*

In [ ]:
%%writefile algoritmos.py
import heapq
from config import LINHAS, COLUNAS, DIRECOES, CUSTO_MOVIMENTO, E_MAX, bateria

def astar(grid, start, goal):

    open_list = []
    heapq.heappush(open_list, (0, start))

    came = {}
    g = {start: 0}

    closed = set()

    while open_list:

        _, current = heapq.heappop(open_list)

        if current in closed:
            continue

        closed.add(current)

        if current == goal:

            path = []

            while current in came:
                path.append(current)
                current = came[current]

            path.append(start)
            path.reverse()

            return path

        for n in neighbors(grid, current):

            tentative = g[current] + custo(grid, n[0], n[1])

            if tentative > bateria:
                continue

            if n not in g or tentative < g[n]:

                g[n] = tentative
                f = tentative + heuristic(n, goal)

                heapq.heappush(open_list, (f, n))
                came[n] = current

    return None

# IDA*

In [ ]:
%%writefile algoritmos.py
import heapq
from config import LINHAS, COLUNAS, DIRECOES, CUSTO_MOVIMENTO, E_MAX, bateria

def ida_star(grid, start, goal):

    bound = heuristic(start, goal)
    path = [start]

    def search(path, g, bound):

        node = path[-1]

        if g > E_MAX:
            return float("inf")

        f = g + heuristic(node, goal)

        if f > bound:
            return f

        if node == goal:
            return path.copy()

        minimum = float("inf")

        for n in neighbors(grid, node):

            if n in path:
                continue

            path.append(n)

            t = search(path, g + 1, bound)

            if isinstance(t, list):
                return t

            if t < minimum:
                minimum = t

            path.pop()

        return minimum

    while True:

        t = search(path, 0, bound)

        if isinstance(t, list):
            return t

        if t == float("inf"):
            return None

        bound = t